In [ ]:
# STEP 1: Install & Import Libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# STEP 2: Upload Your CSV File (Signal Data)
uploaded = files.upload()

# Load CSV
df = pd.read_csv(next(iter(uploaded)))
print("Original shape:", df.shape)

# Drop empty columns (like 'Unnamed: 32')
df = df.dropna(axis=1, how='all')
print("After dropping empty columns:", df.shape)


Saving preprocessed_eeg_dataset.csv to preprocessed_eeg_dataset.csv
Original shape: (68, 21)
After dropping empty columns: (68, 21)


In [ ]:
# STEP 3: Normalize Data
scaler = StandardScaler()
data = scaler.fit_transform(df.values)
data = torch.tensor(data, dtype=torch.float32)

# Parameters
input_dim = data.shape[1]
latent_dim = 100
batch_size = 64
num_epochs = 500


In [ ]:
# STEP 4: Create DataLoader
dataloader = DataLoader(TensorDataset(data), batch_size=batch_size, shuffle=True)


In [ ]:
# STEP 5: Define Generator and Discriminator

class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
# STEP 6: Instantiate Models and Optimizers
G = Generator(latent_dim, input_dim).to(device)
D = Discriminator(input_dim).to(device)

criterion = nn.BCELoss()
optimizer_G = torch.optim.Adam(G.parameters(), lr=0.0002)
optimizer_D = torch.optim.Adam(D.parameters(), lr=0.0002)


In [ ]:
# STEP 7: Train the GAN

for epoch in range(num_epochs):
    for real_batch, in dataloader:
        real_batch = real_batch.to(device)
        batch_size = real_batch.size(0)

        # Train Discriminator
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_batch = G(noise).detach()

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        D_loss_real = criterion(D(real_batch), real_labels)
        D_loss_fake = criterion(D(fake_batch), fake_labels)
        D_loss = D_loss_real + D_loss_fake

        optimizer_D.zero_grad()
        D_loss.backward()
        optimizer_D.step()

        # Train Generator
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_batch = G(noise)
        G_loss = criterion(D(fake_batch), real_labels)

        optimizer_G.zero_grad()
        G_loss.backward()
        optimizer_G.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | D Loss: {D_loss.item():.4f} | G Loss: {G_loss.item():.4f}")


Epoch 50/500 | D Loss: 0.8324 | G Loss: 0.7877
Epoch 100/500 | D Loss: 1.2961 | G Loss: 1.5869
Epoch 150/500 | D Loss: 0.6750 | G Loss: 2.3444
Epoch 200/500 | D Loss: 1.9169 | G Loss: 1.4481
Epoch 250/500 | D Loss: 0.7740 | G Loss: 2.1974
Epoch 300/500 | D Loss: 0.5596 | G Loss: 1.3832
Epoch 350/500 | D Loss: 1.0256 | G Loss: 0.8967
Epoch 400/500 | D Loss: 0.3995 | G Loss: 1.3185
Epoch 450/500 | D Loss: 0.3203 | G Loss: 1.5962
Epoch 500/500 | D Loss: 0.4777 | G Loss: 1.9012


In [ ]:
# STEP 8: Generate and Save New Signals

def generate_and_save(num_samples=1000, filename='generated_signals.csv'):
    G.eval()
    with torch.no_grad():
        noise = torch.randn(num_samples, latent_dim).to(device)
        generated = G(noise).cpu().numpy()
        generated = scaler.inverse_transform(generated)
        pd.DataFrame(generated, columns=df.columns).to_csv(filename, index=False)
        files.download(filename)

generate_and_save(1000)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>